# Session 5 lab · Attention, ViT & CLIP

The code behind the slides. Each cell shows **one idea** in a few lines:

1. run it and look at the result;
2. change a value marked **`· try`** (the values after it are good ones to try) and run it again;
3. read *What to notice* under the cell.

| Part | Ideas |
|---|---|
| **A · Attention** | scores, softmax and the weighted sum by hand · the same in PyTorch · the N × N cost |
| **B · Vision Transformer** | a patch embedding is a convolution · counting tokens in a real ViT · where it looks · why positions are needed |
| **C · Learning without labels** | CLIP image embeddings · cosine similarity · nearest neighbours · two views of one photo |
| **D · CLIP & zero-shot** | classify with sentences · change a word · a note that fools CLIP · your own photo |

Nothing to hand in, nothing to upload — it runs on free Colab. Parts B–D download two models once (about 950 MB together).

In [ ]:
# Setup — run this cell first
import numpy as np, matplotlib.pyplot as plt, urllib.request, os, warnings
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
warnings.filterwarnings('ignore')

def photo(name):                                   # the photos from the slides (CC0, Wikimedia Commons)
    if not os.path.exists(name):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/INCORTX/INCORTX.github.io/cv-next/ComputerVision/session-05/img/' + name, name)
    return Image.open(name).convert('RGB')

NAMES = ['dog', 'dog2', 'cat', 'cat2', 'elephants', 'apple', 'bicycle']
photos = {k: photo(k + '.jpg') for k in NAMES}
torch.manual_seed(0)
print('ready ·', ', '.join(f'{k} {im.size[0]}×{im.size[1]}' for k, im in photos.items()))

## A · Attention

Each token makes a **query**, a **key** and a **value**. Score = q · k; softmax turns the scores into weights that sum to 1; the output is the weighted sum of the values.

In [ ]:
# A1 · Attention by hand: the Try it on slide 10
k = 1                         # scales the query · try 2, 0
q = np.array([1, 2]) * k
K = np.array([[1, 0], [0, 1], [1, 1]])
V = np.array([[1, 0], [0, 2], [2, 2]])

scores = K @ q                                    # one dot product per token
weights = np.exp(scores - scores.max()); weights /= weights.sum()   # softmax
print('scores  ', scores)
print('weights ', weights.round(3), ' sum', weights.sum().round(3))
print('output  ', (weights @ V).round(3))

**What to notice:** scores 1, 2, 3 give weights 0.090, 0.245, 0.665 and the output [1.421, 1.82] — slide 10 rounds the weights to 0.09 / 0.24 / 0.67 and gets [1.43, 1.82]. k = 2 (q = [2, 4]) sharpens the weights — most goes to token 3; k = 0 (q = [0, 0]) makes every score 0, so the weights are equal and the output is the plain average.

In [ ]:
# A2 · The same in PyTorch — which also divides by √d (slide 7)
scale = 'sqrt_d'    # 'sqrt_d' (what Transformers use) or 'none' (slide 10) · try 'none'

Q = torch.tensor([[1., 2]]); Kt = torch.tensor([[1., 0], [0, 1], [1, 1]]); Vt = torch.tensor([[1., 0], [0, 2], [2, 2]])
out = F.scaled_dot_product_attention(Q[None], Kt[None], Vt[None], scale=None if scale == 'sqrt_d' else 1.0)
w = torch.softmax(Q @ Kt.T / (2 ** 0.5 if scale == 'sqrt_d' else 1.0), -1)
print('weights', w.numpy().round(3), '→ output', out[0].numpy().round(3))

**What to notice:** PyTorch divides the scores by √d = √2 unless told otherwise: the weights spread to 0.140, 0.284, 0.576 (slide 7). With `scale = 'none'` you get A1's answer exactly.

In [ ]:
# A3 · Every token looks at every token: the size of one attention table (slide 8)
image = 224     # · try 384, 1024
patch = 16      # · try 1, 32

tokens = (image // patch) ** 2
print(f'{image} × {image} image, {patch} × {patch} patches → {tokens:,} tokens → {tokens ** 2:,} scores per head per layer')

**What to notice:** 196 tokens give 38,416 scores; patch 1 (pixel by pixel) gives 2,517,630,976 — that is why a ViT cuts the image into patches first. Doubling the image side multiplies the tokens by 4 and the scores by 16.

## B · Vision Transformer

Cut the photo into 16 × 16 patches, turn each into a vector (a **token**), add a position, and run attention + MLP blocks. Here: a real ViT-B/16 — the DINO model behind the slides' attention maps.

In [ ]:
# B1 · A patch embedding is a convolution with stride = kernel (slide 14)
patch = 16     # · try 32, 8

embed = nn.Conv2d(3, 768, kernel_size=patch, stride=patch)
x = torch.zeros(1, 3, 224, 224)
y = embed(x)                                  # 1 × 768 × 14 × 14
tokens = y.flatten(2).transpose(1, 2)         # 1 × 196 × 768: one row per patch
print('conv output', tuple(y.shape), '→ tokens', tuple(tokens.shape), '· parameters', f'{sum(p.numel() for p in embed.parameters()):,}')

**What to notice:** ⌊(224 − 16)/16⌋ + 1 = 14 — session 3's output-size formula — so 14 × 14 = 196 tokens of 768, and (16·16·3 + 1)·768 = 590,592 parameters, the number read from the real model on slide 14. Patch 32 gives 49 tokens; patch 8 gives 784.

In [ ]:
# B2 · A real ViT-B/16: count its tokens and weights (slide 17)
from transformers import ViTModel
import torchvision.transforms as T
vit = ViTModel.from_pretrained('facebook/dino-vitb16', attn_implementation='eager', add_pooling_layer=False).eval()
size = 224      # · try 384

prep = T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
with torch.no_grad():
    out = vit(pixel_values=prep(photos['dog'])[None], interpolate_pos_encoding=True)
print('tokens out:', tuple(out.last_hidden_state.shape), ' (1 photo × tokens × 768)')
print(f'weights: {sum(p.numel() for p in vit.parameters()):,} in {vit.config.num_hidden_layers} layers')

**What to notice:** 197 tokens at 224 — 196 patches + the [CLS] token — and 85,798,656 weights, as on slide 17. At 384 the same weights read 577 tokens: only the position embedding is stretched.

In [ ]:
# B3 · Where does it look? The [CLS] token's attention in the last layer (slide 18)
which = 'dog'   # · try 'elephants', 'cat', 'bicycle'

with torch.no_grad():
    out = vit(pixel_values=prep(photos[which])[None], output_attentions=True, interpolate_pos_encoding=True)
side = int((out.attentions[-1].shape[-1] - 1) ** 0.5)
att = out.attentions[-1][0, :, 0, 1:].mean(0).reshape(side, side)     # 12 heads averaged, [CLS] → every patch
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(photos[which].resize((size, size))); ax[0].set_title(which)
ax[1].imshow(photos[which].resize((size, size)).convert('L'), cmap='gray'); ax[1].imshow(att, cmap='inferno', alpha=0.6, extent=(0, size, size, 0))
ax[1].set_title('[CLS] attention, last layer')
for a in ax: a.axis('off')
plt.show()

**What to notice:** the bright patches sit on the animal — the dog's head, the trunk and the calves, the cat's face — although nobody gave this model a label (DINO, part C). On the bicycle it is spread more widely: there is no single 'face' to look at.

In [ ]:
# B4 · Attention ignores order — unless you add positions (slide 15)
use_position = False   # · try True

rng = np.random.default_rng(5)
X = rng.normal(size=(196, 48))                                       # 196 tokens of 48 numbers
Wq, Wk, Wv = (rng.normal(0, 48 ** -0.5, (48, 8)) for _ in range(3))
pos = rng.normal(0, 0.5, (196, 48))                                  # one vector per place, not per patch
def layer(X):
    s = (X @ Wq) @ (X @ Wk).T / 8 ** 0.5; w = np.exp(s - s.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
    return w @ (X @ Wv)
perm = rng.permutation(196)
a = layer(X + pos if use_position else X)[perm]                       # run, then shuffle
b = layer(X[perm] + pos if use_position else X[perm])                 # shuffle, then run
print('largest difference:', np.abs(a - b).max())

**What to notice:** without positions the difference is 0 (up to about 10⁻¹⁵): shuffling before or after attention gives the same tokens, so attention sees a bag of patches. With a position vector for each place, the shuffle changes the answer.

## C · Learning without labels

An **embedding** turns a photo into a point; **cosine similarity** compares directions. Here the points come from CLIP's image encoder: 512 numbers per photo.

In [ ]:
# C1 · Embed the photos with CLIP and compare them (slides 24–25)
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').eval()
proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

with torch.no_grad():
    E = clip.get_image_features(**proc(images=[photos[k] for k in NAMES], return_tensors='pt'))
E = E / E.norm(dim=-1, keepdim=True)              # length 1, so cosine = dot product
C = (E @ E.T).numpy()
print('        ' + ' '.join(f'{k[:6]:>6s}' for k in NAMES))
for k, row in zip(NAMES, C): print(f'{k:8s}' + ' '.join(f'{v:6.2f}' for v in row))

**What to notice:** 1.00 on the diagonal (every photo matches itself); the two cats and the dogs score high with each other, the apple and the bicycle lower. All values are positive: CLIP's photo embeddings all point roughly the same way, so compare them with each other, not with 0.

In [ ]:
# C2 · Search by example: the nearest neighbours (slide 27)
query = 'cat'   # · try 'dog', 'apple'

i = NAMES.index(query)
order = np.argsort(-C[i])
print(f'query: {query}')
for j in order[1:5]: print(f'   {NAMES[j]:10s} {C[i, j]:.2f}')

**What to notice:** the cat finds the other cat first (0.83). The dog on grass finds the cat **on grass** (0.74) before the dog on a road (0.72): one embedding for the whole photo also carries the background — what slide 27 warns about.

In [ ]:
# C3 · Two random views of one photo stay close (slide 26)
crop_scale = 0.4    # smallest crop, as a fraction of the photo · try 0.1, 0.8

aug = T.Compose([T.RandomResizedCrop(224, scale=(crop_scale, 0.9)), T.RandomHorizontalFlip(), T.ColorJitter(0.4, 0.4, 0.4, 0.1)])
torch.manual_seed(0)
views = [aug(photos['dog']) for _ in range(2)]
with torch.no_grad():
    Vv = clip.get_image_features(**proc(images=views, return_tensors='pt'))
Vv = Vv / Vv.norm(dim=-1, keepdim=True)
print(f'view 1 vs view 2 {float(Vv[0] @ Vv[1]):.2f} · vs cat {float(Vv[0] @ E[NAMES.index("cat")]):.2f} · vs bicycle {float(Vv[0] @ E[NAMES.index("bicycle")]):.2f}')
fig, ax = plt.subplots(1, 2, figsize=(6, 3)); [a.imshow(v) or a.axis('off') for a, v in zip(ax, views)]; plt.show()

**What to notice:** the two views stay closer to each other than to any other photo — the positive pair of contrastive learning. Tiny crops (0.1) can cut away the dog and push the pair apart: augmentation must keep what the photo is about.

## D · CLIP & zero-shot

Write the classes as sentences, embed them with CLIP's text encoder, and pick the sentence with the largest cosine. No training.

In [ ]:
# D1 · Zero-shot: the prompt is the classifier (slide 33)
which = 'dog'    # · try 'cat', 'elephants', 'bicycle'
classes = ['dog', 'cat', 'apple', 'elephant', 'bicycle']

prompts = [f'a photo of a{"n" if c[0] in "aeiou" else ""} {c}' for c in classes]
def zero_shot(img, texts):
    with torch.no_grad():
        o = clip(**proc(text=texts, images=img, return_tensors='pt', padding=True))
        cos = (o.logits_per_image[0] / clip.logit_scale.exp()).numpy()   # the cosines
        return cos, o.logits_per_image.softmax(-1)[0].numpy()             # softmax of 100 × cosine
cos, p = zero_shot(photos[which], prompts)
for t, c, q in sorted(zip(prompts, cos, p), key=lambda r: -r[2]): print(f'{t:26s} cos {c:.3f}  p {q:.3f}')

**What to notice:** the cosines are close (about 0.2–0.3), yet one class gets nearly all the probability: CLIP multiplies the cosines by 100 before softmax. Add a class by adding a sentence — nothing is retrained.

In [ ]:
# D2 · Change a word, change the answer (slide 34)
which = 'cat'    # · try 'dog'
prompts = ['cat', 'a photo of a cat', 'a photo of a tabby cat', 'a photo of a Thai cat', 'a drawing of a cat', 'a photo of a tiger']   # change the wording and run again

cos, p = zero_shot(photos[which], prompts)
for t, q in sorted(zip(prompts, p), key=lambda r: -r[1]): print(f'{t:26s} {q:.2f}')

**What to notice:** on the cat, “a photo of a Thai cat” wins (0.71) and the bare word “cat” gets 0.02. On the dog this list has no dog in it, so CLIP picks the closest sentence it was given — “a photo of a tiger” (0.64): the prompts decide what CLIP can answer. Test your wording on validation photos before you trust it.

In [ ]:
# D3 · Text in the picture fools CLIP (slide 35)
word = 'iPod'    # · try 'dog', 'cat'

img = photos['apple'].copy(); d = ImageDraw.Draw(img); W, H = img.size
font = ImageFont.load_default(size=60)
x, y = int(W * 0.38), int(H * 0.42); bb = d.textbbox((x, y), word, font=font)
d.rectangle((bb[0] - 12, bb[1] - 10, bb[2] + 12, bb[3] + 12), fill=(250, 248, 235)); d.text((x, y), word, fill=(20, 20, 40), font=font)
cos, p = zero_shot(img, ['a photo of an apple', 'a photo of a dog', 'a photo of a cat', 'a photo of an iPod'])
print(dict(zip(['apple', 'dog', 'cat', 'iPod'], p.round(2))))
plt.imshow(img); plt.axis('off'); plt.show()

**What to notice:** a paper note reading “iPod” turns the apple into an iPod for CLIP (0.88 here; 0.92 on slide 35, whose note uses another font); “dog” pulls the answer towards dog. CLIP learned from photo captions, and written words in a photo often *are* the caption — so it reads before it looks.

In [ ]:
# D4 · Your own photo
my_photo = 'elephants.jpg'     # upload a photo in the Files panel (left) and put its name here
my_classes = ['one elephant', 'two elephants', 'three elephants', 'four elephants', 'five elephants']

img = Image.open(my_photo).convert('RGB')
cos, p = zero_shot(img, [f'a photo of {c}' for c in my_classes])
for c, q in sorted(zip(my_classes, p), key=lambda r: -r[1]): print(f'{c:20s} {q:.2f}')
plt.imshow(img); plt.axis('off'); plt.show()

**What to notice:** there are four elephants in this photo; CLIP's top answer is “three” (0.31) and “four” comes fourth (0.14) — it cannot count (slide 35). Try your own photo and classes: where does zero-shot work, and where would your project need training or a detector (session 8)?

---
Back to the slides: [session 5 deck](https://classes.incortx.com/ComputerVision/session-05/slides.html) · exam-style questions: [practice bank](https://classes.incortx.com/ComputerVision/practice/#s5).